# 🎸 Stems para remezclar (Colab · GPU T4)

Separa cada tema en stems **sumables** (la suma de todos = la mezcla, por resta exacta), los afina a tu referencia **antes** de separar y deja todo listo para Logic o Ableton.

**Cómo se usa**
1. Sube tus audios (FLAC, WAV o MP3) a `Drive/stems/entrada`.
2. Activa la GPU: *Entorno de ejecución › Cambiar tipo de entorno de ejecución › T4 GPU*.
3. Ajusta **solo** la celda de configuración de abajo.
4. *Entorno de ejecución › Ejecutar todo*.
5. Resultados en `Drive/stems/salida/<tema>/`.

Los temas ya terminados se saltan solos. Para rehacer uno, borra a mano su carpeta de salida.

## ⚙️ Configuración

In [ ]:
#@title ⚙️ Configuración (lo único que se ajusta) { display-mode: "form" }
#@markdown ### 1 · Modo
#@markdown **karaoke** = solo voz principal / coros / instrumental (rápido)
#@markdown <br>**completo** = todos los stems: voces, batería, bajo, guitarras, piano, otros (lento)
MODO = "completo"  #@param ["karaoke", "completo"]

#@markdown ### 2 · Guitarras (solo en modo completo; en karaoke se ignoran y no se descargan sus modelos)
#@markdown Modelo que separa las guitarras del resto (compara de oído y fija el ganador):
MODELO_GUITARRA = "becruily"  #@param ["becruily", "chencfd"]
#@markdown Orden: "punteo primero" separa el solo antes que acústica/eléctrica (recomendado para solos clean):
ORDEN_GUITARRAS = "punteo primero"  #@param ["punteo primero", "timbre primero"]
#@markdown Separar guitarras acústicas de eléctricas (descarga una vez un modelo de 1,37 GB):
SEPARAR_ACUSTICA_ELECTRICA = True  #@param {type:"boolean"}

#@markdown ### 3 · Afinación (se mide y se corrige ANTES de separar)
#@markdown Activar la corrección de afinación:
AFINAR = True  #@param {type:"boolean"}
#@markdown Frecuencia de referencia destino (La4):
REFERENCIA = "440 Hz"  #@param ["432 Hz", "440 Hz", "442 Hz", "444 Hz", "otra (usar el deslizador)"]
#@markdown Solo si elegiste "otra":
OTRA_REFERENCIA_HZ = 440  #@param {type:"slider", min:400, max:480, step:0.5}
#@markdown Solo corrige si la desviación supera este umbral (cents):
UMBRAL_CENTS = 1.5  #@param [0.5, 1.0, 1.5, 2.0, 3.0] {type:"raw"}

#@markdown ### 4 · Formato de salida
#@markdown FLAC 24-bit para el DAW (la suma de stems = la mezcla); MP3 solo para escuchar (rompe la suma exacta):
FORMATO = "FLAC 24-bit"  #@param ["FLAC 24-bit", "MP3 320"]
#@markdown Frecuencia de muestreo de los archivos (48 kHz = remuestreo de alta calidad al final):
FRECUENCIA = "44.1 kHz"  #@param ["44.1 kHz", "48 kHz"]
#@markdown Clic de referencia (llega en la fase 2; por ahora no hace nada):
CLIC = "ambos"  #@param ["no", "estable", "exacto", "ambos"]

#@markdown ### 5 · Carpetas en Drive (normalmente no hay que tocarlas)
CARPETA_ENTRADA = "/content/drive/MyDrive/stems/entrada"  #@param {type:"string"}
CARPETA_SALIDA = "/content/drive/MyDrive/stems/salida"  #@param {type:"string"}
CARPETA_MODELOS = "/content/drive/MyDrive/stems/modelos"  #@param {type:"string"}

# --- Valores derivados (no editar) ---
if REFERENCIA.startswith("otra"):
    DESTINO_HZ = float(OTRA_REFERENCIA_HZ)
else:
    DESTINO_HZ = float(REFERENCIA.split()[0])
if not AFINAR:
    DESTINO_HZ = None
UMBRAL_CENTS = float(UMBRAL_CENTS)
# minutos de proceso por minuto de canción en una T4 (aproximado; el resumen final muestra el real)
MIN_POR_MIN_AUDIO = 1.0 if MODO == "karaoke" else (2.5 if SEPARAR_ACUSTICA_ELECTRICA else 2.0)


def resumen_configuracion():
    L = ["Configuración elegida:"]
    if MODO == "karaoke":
        L.append("  · Modo karaoke: 01 Voz principal, 02 Coros, 03 Instrumental (rápido)")
        L.append("    (los parámetros de guitarras se ignoran y no se descargan sus modelos)")
    else:
        L.append("  · Modo completo: voces, batería, bajo, guitarras, piano y otros (lento)")
        L.append(f"  · Guitarras: modelo {MODELO_GUITARRA} · orden «{ORDEN_GUITARRAS}» · "
                 f"acústica/eléctrica: {'sí' if SEPARAR_ACUSTICA_ELECTRICA else 'no'}")
    if DESTINO_HZ is None:
        L.append("  · Afinación: no se mide ni se corrige")
    else:
        L.append(f"  · Afinación: a {DESTINO_HZ:g} Hz, solo si la desviación supera ±{UMBRAL_CENTS:g} ¢")
    L.append(f"  · Salida: {FORMATO} a {FRECUENCIA}" + (" (⚠ MP3 rompe la suma exacta)" if FORMATO.startswith("MP3") else ""))
    L.append(f"  · Clic: {CLIC} (fase 2)")
    L.append(f"  · Tiempo estimado: ~{MIN_POR_MIN_AUDIO:g} min por cada minuto de canción "
             f"(un tema de 4 min ≈ {4 * MIN_POR_MIN_AUDIO:.0f} min), más la descarga de modelos la primera vez")
    return "\n".join(L)


print(resumen_configuracion())


## 1 · Preparar
Revisa la GPU, monta Drive, crea las carpetas y revisa el espacio libre.

In [ ]:
#@title 1 · Preparar: GPU, Drive, carpetas y espacio
import os, sys, json, math, time, shutil, subprocess, traceback
from pathlib import Path


class Detenido(Exception):
    """Error con mensaje claro para ti: detiene "Ejecutar todo" en esta celda."""


# --- GPU (se revisa con nvidia-smi, antes de instalar nada) ---
_gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True) if shutil.which("nvidia-smi") else None
if _gpu is None or _gpu.returncode != 0 or "GPU" not in _gpu.stdout:
    raise Detenido(
        "No hay GPU activa.\n"
        "Actívala así: menú Entorno de ejecución > Cambiar tipo de entorno de ejecución > "
        "Acelerador por hardware: T4 GPU > Guardar.\n"
        "Después: Entorno de ejecución > Ejecutar todo.")
print("✓", _gpu.stdout.strip())

# --- Drive y carpetas ---
from google.colab import drive
drive.mount("/content/drive")

ENTRADA = Path(CARPETA_ENTRADA)
SALIDA = Path(CARPETA_SALIDA)
MODELOS = Path(CARPETA_MODELOS)
TRABAJO = Path("/content/trabajo")  # temporales locales, fuera de Drive
for _c in (ENTRADA, SALIDA, MODELOS, TRABAJO):
    _c.mkdir(parents=True, exist_ok=True)

EXTENSIONES = {".flac", ".wav", ".mp3"}
TEMAS = sorted(p for p in ENTRADA.iterdir() if p.is_file() and p.suffix.lower() in EXTENSIONES)
print(f"✓ Entrada: {ENTRADA} ({len(TEMAS)} archivos de audio)")
for p in TEMAS:
    print("   ·", p.name)
if not TEMAS:
    raise Detenido(f"No hay audios (FLAC, WAV o MP3) en {ENTRADA}. Súbelos ahí y vuelve a ejecutar.")


# --- Espacio en Drive ---
def duracion_segundos(ruta):
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                        "-of", "default=nw=1:nk=1", str(ruta)], capture_output=True, text=True)
    try:
        return float(r.stdout.strip())
    except ValueError:
        return 300.0  # si no se puede leer, suponer 5 min


def estimar_mb_tema(segundos):
    """Tamaño aproximado de la carpeta de un tema (stems + mezcla afinada + clics)."""
    n_stems = (3 if MODO == "karaoke" else 9) + 1 + (2 if CLIC != "no" else 0)
    if FORMATO.startswith("MP3"):
        bytes_por_s = 320_000 / 8
    else:
        sr = 48000 if FRECUENCIA.startswith("48") else 44100
        bytes_por_s = sr * 2 * 3 * 0.6  # 24 bit estéreo, FLAC comprime ~40 %
    return n_stems * segundos * bytes_por_s / 1e6


_pendientes = [p for p in TEMAS if not (SALIDA / p.stem / ".proceso.json").exists()]
_duraciones = {p: duracion_segundos(p) for p in _pendientes}
_estimado = sum(estimar_mb_tema(d) for d in _duraciones.values())
if (MODO == "completo" and SEPARAR_ACUSTICA_ELECTRICA
        and not (MODELOS / "bs_roformer_mega_53stem_mvsep.ckpt").exists()):
    _estimado += 1400  # modelo de guitarra acústica (53 stems, 1,37 GB), se descarga una vez
_minutos_audio = sum(_duraciones.values()) / 60
print(f"✓ Tiempo estimado para los pendientes: ~{_minutos_audio * MIN_POR_MIN_AUDIO:.0f} min "
      f"({_minutos_audio:.1f} min de audio × {MIN_POR_MIN_AUDIO:g}; aproximado)")
_libre = shutil.disk_usage(SALIDA).free / 1e6
print(f"✓ Espacio libre en Drive: {_libre / 1000:.1f} GB · "
      f"estimado para {len(_pendientes)} tema(s) pendiente(s): {_estimado / 1000:.2f} GB "
      f"(~{_estimado / max(1, len(_pendientes)):.0f} MB por tema)")
if _libre < 2000:
    print("⚠ ATENCIÓN: quedan menos de 2 GB libres en Drive.")
if _estimado > _libre:
    raise Detenido("No alcanza el espacio en Drive para los temas pendientes. "
                   "Libera espacio o deja menos archivos en la carpeta de entrada.")


## 2 · Instalar
audio-separator **fijo en 0.44.5** (el arreglo de la celda 6 depende de esa versión), Rubber Band 3 (pylibrb), soxr.

In [ ]:
#@title 2 · Instalar programas (2–4 min la primera vez)
# ⚠ audio-separator queda FIJO en 0.44.5. El arreglo de la celda 6 (monkeypatch de
#   MelBandRoformer) depende de esa versión exacta: si lo actualizas, puede romperse.
_paquetes = ["audio-separator[gpu]==0.44.5", "pylibrb==0.1.2", "soxr", "huggingface_hub",
             "audioread"]  # audioread: audio-separator lo importa y librosa 1.x ya no lo trae
_r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_paquetes], capture_output=True, text=True)
if _r.returncode != 0:
    print(_r.stdout[-3000:], _r.stderr[-3000:])
    raise Detenido("Falló la instalación de paquetes (ver arriba). Reinicia el entorno y vuelve a ejecutar todo.")
if not shutil.which("ffmpeg"):
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)

from importlib.metadata import version as _version
print("✓ audio-separator", _version("audio-separator"), "· pylibrb", _version("pylibrb"))


## 3 · Utilidades
Borrado seguro (solo dentro de las carpetas de trabajo, nunca con comodines, jamás en `entrada/`), lectura con **una sola decodificación** y escritura FLAC 24-bit / MP3.

In [ ]:
#@title 3 · Utilidades: borrado seguro, lectura y escritura de audio
import numpy as np
import soundfile as sf
import soxr

SR = 44100  # la separación y la afinación trabajan siempre a 44,1 kHz
SR_SALIDA = 48000 if FRECUENCIA.startswith("48") else 44100
ES_MP3 = FORMATO.startswith("MP3")
EXT = ".mp3" if ES_MP3 else ".flac"


# ---------- Borrado seguro: la ÚNICA forma de borrar en este notebook ----------
def _dentro(ruta, base):
    try:
        ruta.relative_to(base)
        return True
    except ValueError:
        return False


def borrar_seguro(ruta, tema_en_curso=None):
    """Borra UN archivo o UNA carpeta con nombre exacto (nunca comodines).
    Solo se permite dentro de /content/trabajo/ o de salida/<tema en curso>/.
    Jamás toca la carpeta de entrada ni la de modelos."""
    texto = str(ruta)
    if any(c in texto for c in "*?[]"):
        raise PermissionError(f"borrar_seguro: no se aceptan comodines: {texto}")
    r = Path(ruta).resolve()
    for prohibida in (ENTRADA.resolve(), MODELOS.resolve()):
        if _dentro(r, prohibida) or _dentro(prohibida, r):
            raise PermissionError(f"borrar_seguro: {r} está en (o contiene) una carpeta protegida: {prohibida}")
    permitidas = [TRABAJO.resolve()]
    if tema_en_curso:
        permitidas.append((SALIDA / tema_en_curso).resolve())
    if not any(r != p and _dentro(r, p) for p in permitidas):
        raise PermissionError(f"borrar_seguro: {r} está fuera de las carpetas de trabajo permitidas")
    if not r.exists() and not r.is_symlink():
        return
    if r.is_dir() and not r.is_symlink():
        shutil.rmtree(r)
    else:
        r.unlink()


# ---------- Avance visible (barras, línea fija y progreso.txt en Drive) ----------
def _hms(segundos):
    segundos = int(max(0, segundos))
    return f"{segundos // 3600}:{segundos % 3600 // 60:02d}:{segundos % 60:02d}" if segundos >= 3600 \
        else f"{segundos // 60}:{segundos % 60:02d}"


class Progreso:
    """Tres barras que se actualizan en el mismo lugar (tqdm: temas, etapas del tema, segmentos),
    una línea fija de texto (por si el widget no se ve en el celular) y Drive/stems/salida/progreso.txt,
    que se reescribe en cada etapa para revisar el avance desde otro dispositivo."""
    INTERVALO_S = 1.0         # refresco mínimo de la línea fija
    INTERVALO_ARCHIVO_S = 20  # dentro de una etapa, reescribir progreso.txt cada tanto

    def __init__(self, n_temas, archivo=None, mostrar=True):
        self.n_temas, self.archivo, self.mostrar = n_temas, archivo, mostrar
        self.i_tema, self.tema, self.etapas_total, self.i_etapa, self.etapa = 0, "", 1, 0, ""
        self.seg, self.seg_total = 0, 0
        self.t0 = self.t_tema = self.t_etapa = time.time()
        self._t_linea = self._t_archivo = 0.0
        self.barras = self.linea = None
        if mostrar:
            try:
                from tqdm.auto import tqdm
                fmt = "{desc} {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]"
                self.barras = [tqdm(total=n_temas, desc="Temas", bar_format=fmt),
                               tqdm(total=1, desc="Etapas", bar_format=fmt),
                               tqdm(total=1, desc="Segmentos", bar_format=fmt)]
            except Exception:
                self.barras = None
            try:
                from IPython import get_ipython
                from IPython.display import display
                if get_ipython() is not None:  # solo dentro de Colab/Jupyter
                    self.linea = display({"text/plain": "▶ preparando…"}, raw=True, display_id=True)
            except Exception:
                self.linea = None

    # --- eventos ---
    def nuevo_tema(self, i, nombre, etapas_total):
        self.i_tema, self.tema, self.etapas_total, self.i_etapa, self.etapa = i, nombre, max(1, etapas_total), 0, ""
        self.seg = self.seg_total = 0
        self.t_tema = time.time()
        if self.barras:
            self.barras[0].n = i - 1; self.barras[0].refresh()
            self.barras[1].reset(total=self.etapas_total); self.barras[1].set_description(f"Etapas · {nombre[:24]}")
            self.barras[2].reset(total=1)
        self._actualizar(forzar=True)

    def nueva_etapa(self, nombre):
        self.i_etapa = min(self.i_etapa + 1, self.etapas_total)
        self.etapa, self.seg, self.seg_total = nombre, 0, 0
        self.t_etapa = time.time()
        if self.barras:
            self.barras[1].n = self.i_etapa - 1; self.barras[1].refresh()
            self.barras[2].reset(total=1); self.barras[2].set_description(f"Segmentos · {nombre[:24]}")
        self._actualizar(forzar=True)

    def segmento(self, k, total):
        self.seg, self.seg_total = k, total
        if self.barras:
            if self.barras[2].total != total:
                self.barras[2].reset(total=total)
            self.barras[2].n = k; self.barras[2].refresh()
        self._actualizar()

    def fin_tema(self):
        if self.barras:
            self.barras[1].n = self.etapas_total; self.barras[1].refresh()
            self.barras[0].n = self.i_tema; self.barras[0].refresh()
        self.i_etapa = self.etapas_total
        self._actualizar(forzar=True)

    def cerrar(self, resumen):
        if self.barras:
            for b in self.barras:
                b.close()
        if self.linea:
            self.linea.update({"text/plain": "✓ Terminado. Resumen abajo y en salida/progreso.txt"}, raw=True)
        self._escribir("PROCESO TERMINADO · " + time.strftime("%Y-%m-%d %H:%M:%S") + "\n\n" + resumen)

    # --- salida ---
    def porcentaje_tema(self):
        dentro = (self.seg / self.seg_total) if self.seg_total else 0.0
        return 100.0 * min(1.0, (max(self.i_etapa - 1, 0) + dentro) / self.etapas_total)

    def texto(self):
        ahora = time.time()
        pct = self.porcentaje_tema()
        trans = ahora - self.t_tema
        resta = trans * (100 - pct) / pct if pct > 1 else None
        seg = f" · segmento {self.seg}/{self.seg_total}" if self.seg_total else ""
        return (f"▶ Tema {self.i_tema}/{self.n_temas} «{self.tema}» · etapa {self.i_etapa}/{self.etapas_total}: "
                f"{self.etapa}{seg} · {pct:.0f} % · {_hms(trans)} transcurrido"
                + (f" · ~{_hms(resta)} restante" if resta is not None else "")
                + f" · total {_hms(ahora - self.t0)}")

    def _actualizar(self, forzar=False):
        ahora = time.time()
        if self.linea and (forzar or ahora - self._t_linea >= self.INTERVALO_S):
            self.linea.update({"text/plain": self.texto()}, raw=True)
            self._t_linea = ahora
        if forzar or ahora - self._t_archivo >= self.INTERVALO_ARCHIVO_S:
            self._escribir(f"Actualizado: {time.strftime('%Y-%m-%d %H:%M:%S')}\n{self.texto()}\n")
            self._t_archivo = ahora

    def _escribir(self, contenido):
        if not self.archivo:
            return
        try:  # se escribe en un .parcial y se renombra: nunca queda a medio escribir
            parcial = Path(str(self.archivo) + ".parcial")
            parcial.write_text(contenido, encoding="utf-8")
            os.replace(parcial, self.archivo)
        except OSError as e:
            print("  (no pude escribir progreso.txt:", e, ")")


PROGRESO = None  # lo crea la celda 9


def avisar_segmento(k, total):
    if PROGRESO is not None:
        PROGRESO.segmento(k, total)


# ---------- Tiempos por etapa ----------
class Etapa:
    """Mide el tiempo de una etapa, la anuncia al avance y deja UNA línea al terminar."""
    def __init__(self, nombre, tiempos=None):
        self.nombre, self.tiempos = nombre, tiempos

    def __enter__(self):
        self.t0 = time.time()
        if PROGRESO is not None:
            PROGRESO.nueva_etapa(self.nombre)
        else:
            print(f"  ▸ {self.nombre}…", flush=True)
        return self

    def __exit__(self, *exc):
        dt = time.time() - self.t0
        if self.tiempos is not None:
            self.tiempos[self.nombre] = round(dt, 1)
        if exc[0] is None:
            print(f"    ✓ {self.nombre} ({_hms(dt)})", flush=True)
        return False


# ---------- Audio ----------
def dbfs(valor):
    return 20 * math.log10(max(float(valor), 1e-12))


def pico(x):
    return float(np.max(np.abs(x))) if x.size else 0.0


def leer_audio(ruta):
    """UNA sola decodificación (ffmpeg → float32 a su frecuencia nativa → soxr VHQ a 44,1 kHz).
    Devuelve un array (2, N) float32. Mono se duplica; más de 2 canales se mezcla a estéreo."""
    pr = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "a:0", "-show_entries",
                         "stream=sample_rate", "-of", "default=nw=1:nk=1", str(ruta)],
                        capture_output=True, text=True)
    sr_nativa = int(pr.stdout.strip().splitlines()[0])
    r = subprocess.run(["ffmpeg", "-v", "error", "-i", str(ruta), "-map", "0:a:0", "-vn",
                        "-f", "f32le", "-acodec", "pcm_f32le", "-ac", "2", "-"],
                       capture_output=True)
    if r.returncode != 0:
        raise RuntimeError(f"ffmpeg no pudo leer {ruta.name}: {r.stderr.decode()[-500:]}")
    x = np.frombuffer(r.stdout, dtype=np.float32).reshape(-1, 2).T
    if sr_nativa != SR:
        x = soxr.resample(x.T, sr_nativa, SR, quality="VHQ").T
    return np.ascontiguousarray(x, dtype=np.float32), sr_nativa


def a_frecuencia_salida(x):
    """Remuestreo final de alta calidad (soxr VHQ, lineal: conserva la suma de stems)."""
    if SR_SALIDA == SR:
        return x
    return np.ascontiguousarray(soxr.resample(x.T, SR, SR_SALIDA, quality="VHQ").T, dtype=np.float32)


def escribir_audio(ruta_sin_ext, x, sr):
    """Escribe (2, N) float32. FLAC 24-bit (cuantiza solo aquí) o MP3 320 kbps."""
    ruta = Path(str(ruta_sin_ext) + EXT)
    if ES_MP3:
        r = subprocess.run(["ffmpeg", "-v", "error", "-y", "-f", "f32le", "-ar", str(sr), "-ac", "2",
                            "-i", "-", "-c:a", "libmp3lame", "-b:a", "320k", str(ruta)],
                           input=np.ascontiguousarray(x.T, dtype=np.float32).tobytes(), capture_output=True)
        if r.returncode != 0:
            raise RuntimeError(f"ffmpeg no pudo escribir {ruta.name}: {r.stderr.decode()[-500:]}")
    else:
        sf.write(str(ruta), x.T, sr, subtype="PCM_24", format="FLAC")
    return ruta


def leer_escrito(ruta):
    """Relee un archivo ya escrito (para la prueba de cancelación). Devuelve (2, N) float64."""
    if ruta.suffix == ".flac":
        y, _ = sf.read(str(ruta), dtype="float64", always_2d=True)
        return y.T
    r = subprocess.run(["ffmpeg", "-v", "error", "-i", str(ruta), "-f", "f64le", "-ac", "2", "-"],
                       capture_output=True, check=True)
    return np.frombuffer(r.stdout, dtype=np.float64).reshape(-1, 2).T


def mm_ss(segundos):
    return f"{int(segundos // 60)}:{segundos % 60:04.1f}"

print("✓ Utilidades listas")


## 4 · Medición de afinación
Port a Python del detector de [centrail](https://github.com/Trainmusiq/centrail) (`engine/detect.mjs`).

In [ ]:
#@title 4 · Medición de afinación (port de centrail · engine/detect.mjs)
# Mismo método y parámetros que centrail (docs/especificacion.md §3). No se inventa otro:
# picos espectrales → desviación en cents módulo 100 con matemática circular → histograma
# de 1 ¢/bin + media circular local ±15 ¢; incertidumbre = diferencia entre ventanas
# pares e impares; deriva en 10 segmentos; filtro de persistencia de picos (v1.2).
#
# CASO LÍMITE CONOCIDO (heredado de centrail, se deja igual a propósito): el histograma
# tiene bins de 1 ¢ y la media circular local usa el CENTRO de cada bin, no la desviación
# exacta de cada pico. Un tono perfectamente puro cuyos picos caen todos en el mismo bin
# se mide en el centro de ese bin: error de hasta ±0,5 ¢ (p. ej. una senoidal a +7,000 ¢
# cae en el borde del bin [7, 8) y se mide +7,5 ¢). En música real los picos se reparten
# entre bins vecinos y el error baja a ~0,1–0,2 ¢. Mejora pendiente: refinar con la
# desviación exacta de cada pico, primero en centrail y después portarla aquí.
FFT_N = 32768
MAX_FRAMES = 240              # ventanas repartidas por todo el tema
FMIN, FMAX = 55.0, 5000.0     # rango útil para material armónico
NBINS = 100                   # 1 bin = 1 cent
SEGMENTS = 10                 # curva de deriva
PERSISTENCE_TOL_CENTS = 15
PERSISTENCE_TOP_K = 5
_HANN = 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(FFT_N) / (FFT_N - 1))


def _envolver(d):
    """Lleva cents a [-50, 50) (equivale a ((d % 100) + 150) % 100 - 50 de JS)."""
    return ((np.asarray(d) % 100) + 150) % 100 - 50


def _redondear_js(x):
    """Math.round de JS (las mitades suben)."""
    return np.floor(np.asarray(x) + 0.5).astype(int)


def _picos_ventana(x, inicio, sr):
    """Picos de una ventana: (desviación ¢, peso, frecuencia Hz), del más fuerte al más débil."""
    mono = x[:, inicio:inicio + FFT_N].mean(axis=0) * _HANN
    espectro = np.fft.rfft(mono)
    mitad = FFT_N // 2
    mag = np.abs(espectro[:mitad])
    mag[0] = 0.0
    max_mag = mag[1:].max()
    if max_mag <= 0:
        return np.empty(0), np.empty(0), np.empty(0)
    umbral = max_mag * 0.002
    bin_hz = sr / FFT_N
    i_min = max(2, math.ceil(FMIN / bin_hz))
    i_max = min(mitad - 2, math.floor(FMAX / bin_hz))
    i = np.arange(i_min, i_max + 1)
    m = mag[i]
    es_pico = (m >= umbral) & (m > mag[i - 1]) & (m >= mag[i + 1])
    i, m = i[es_pico], m[es_pico]
    if i.size == 0:
        return np.empty(0), np.empty(0), np.empty(0)
    a, b, c = np.log(mag[i - 1] + 1e-12), np.log(m + 1e-12), np.log(mag[i + 1] + 1e-12)
    den = a - 2 * b + c
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.where(den != 0, 0.5 * (a - c) / den, 0.0)
    p = np.clip(p, -0.5, 0.5)
    f = (i + p) * bin_hz
    dev = _envolver(1200 * np.log2(f / 440.0))
    w = np.sqrt(m / max_mag)
    orden = np.argsort(-w, kind="stable")
    return dev[orden], w[orden], f[orden]


def _coinciden(fa, fb):
    a, b = fa[:PERSISTENCE_TOP_K], fb[:PERSISTENCE_TOP_K]
    if a.size == 0 or b.size == 0:
        return False
    return bool(np.any(np.abs(1200 * np.log2(a[:, None] / b[None, :])) <= PERSISTENCE_TOL_CENTS))


def _estimacion_circular(hist, sum_sin, sum_cos, total_w):
    pico_bin = int(np.argmax(hist))
    pico_dev = pico_bin - 49.5
    dev = np.arange(NBINS) - 49.5
    cerca = np.abs(_envolver(dev - pico_dev)) <= 15
    th = 2 * np.pi * dev / 100
    s = float(np.sum(hist[cerca] * np.sin(th[cerca])))
    c = float(np.sum(hist[cerca] * np.cos(th[cerca])))
    offset = math.atan2(s, c) / (2 * math.pi) * 100
    R = math.hypot(sum_sin, sum_cos) / total_w if total_w > 0 else 0.0
    return offset, R


def medir_afinacion(x, sr=SR):
    """x: (canales, N). Devuelve dict con ref_hz, offset (¢ vs A440), unc, R, deriva, segmentos…"""
    largo = x.shape[1]
    usable = largo - FFT_N
    if usable < 0:
        raise ValueError("El audio es demasiado corto para medir la afinación.")
    n_frames = min(MAX_FRAMES, max(8, usable // (FFT_N // 2)))
    hop = usable / max(1, n_frames - 1)
    inicios = [min(usable, int(_redondear_js(f * hop))) for f in range(n_frames)]
    picos = [_picos_ventana(x, ini, sr) for ini in inicios]

    # persistencia (lógica OR con la ventana vecina; red de seguridad si excluiría todo)
    tonal = []
    for f in range(n_frames):
        if picos[f][0].size == 0:
            tonal.append(False)
            continue
        prev = f > 0 and _coinciden(picos[f][2], picos[f - 1][2])
        sig = f < n_frames - 1 and _coinciden(picos[f][2], picos[f + 1][2])
        tonal.append(prev or sig)
    excluidas = tonal.count(False)
    filtrar = excluidas < n_frames
    if not filtrar:
        excluidas = 0

    hist = np.zeros(NBINS)
    sum_sin = sum_cos = total_w = 0.0
    seg = [[0.0, 0.0, 0.0] for _ in range(SEGMENTS)]
    mitad_hist = [np.zeros(NBINS), np.zeros(NBINS)]
    mitad = [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]
    for f in range(n_frames):
        if filtrar and not tonal[f]:
            continue
        dev, w, _ = picos[f]
        if dev.size == 0:
            continue
        s_i = min(SEGMENTS - 1, int(math.floor(inicios[f] / largo * SEGMENTS)))
        h_i = f % 2
        bins = np.clip(_redondear_js(dev + 49.5), 0, NBINS - 1)
        th = 2 * np.pi * dev / 100
        ws, wc, wt = float(np.sum(w * np.sin(th))), float(np.sum(w * np.cos(th))), float(np.sum(w))
        np.add.at(hist, bins, w)
        np.add.at(mitad_hist[h_i], bins, w)
        sum_sin += ws; sum_cos += wc; total_w += wt
        for acc in (seg[s_i], mitad[h_i]):
            acc[0] += ws; acc[1] += wc; acc[2] += wt

    if total_w <= 0:
        raise ValueError("No se encontraron picos tonales para medir la afinación.")
    offset, R = _estimacion_circular(hist, sum_sin, sum_cos, total_w)

    split_diff = None
    if mitad[0][2] > 0 and mitad[1][2] > 0:
        a, _ = _estimacion_circular(mitad_hist[0], *mitad[0])
        b, _ = _estimacion_circular(mitad_hist[1], *mitad[1])
        split_diff = abs(float(_envolver(a - b)))
    disp = math.sqrt(max(0.0, -2 * math.log(max(1e-6, R)))) * 100 / (2 * math.pi)
    unc_analitica = max(0.1, min(25.0, disp / math.sqrt(max(1, n_frames))))
    unc = max(0.1, split_diff) if split_diff is not None else max(unc_analitica, 8.0)

    segmentos = []
    for s, c, wt in seg:
        if wt <= 0:
            segmentos.append(None)
            continue
        d = math.atan2(s, c) / (2 * math.pi) * 100
        segmentos.append(offset + float(_envolver(d - offset)))
    validos = [s for s in segmentos if s is not None]
    deriva = max(abs(s - offset) for s in validos) if validos else 0.0

    return {"ref_hz": 440.0 * 2 ** (offset / 1200), "offset": offset, "R": R, "unc": unc,
            "split_diff": split_diff, "deriva_max": deriva, "segmentos": segmentos,
            "ventanas": n_frames, "excluidas": excluidas}


def frecuencia_dominante(x, sr=SR):
    """Frecuencia absoluta del pico más fuerte (mediana entre ventanas). Solo para autopruebas."""
    usable = x.shape[1] - FFT_N
    n = min(MAX_FRAMES, max(8, usable // (FFT_N // 2)))
    hop = usable / max(1, n - 1)
    fs = []
    for k in range(n):
        _, _, f = _picos_ventana(x, min(usable, int(round(k * hop))), sr)
        if f.size:
            fs.append(f[0])
    return float(np.median(fs))

print("✓ Medición de afinación lista")


## 5 · Rubber Band R3
Mismas opciones que centrail. Autoprueba: +100 ¢ con error ≤ 0,5 ¢; si pylibrb falla, se compila Rubber Band desde el código fuente.

In [ ]:
#@title 5 · Rubber Band (motor fino R3) + autoprueba
# Mismas opciones que centrail (engine/correct.mjs): ENGINE_FINER | PROCESS_OFFLINE |
# PITCH_HIGH_QUALITY. Sin preservar formantes y sin PITCH_HIGH_CONSISTENCY: es una
# corrección de pocos cents sobre la mezcla completa.
# Motor principal: pylibrb (Rubber Band 3 compilado, desde PyPI). Plan B: compilar el
# programa rubberband 3 desde el código fuente y guardarlo en Drive/stems/modelos/bin.
BLOQUE_RB = 65536
RB_VERSION_FUENTE = "3.3.0"
RB_BINARIO = MODELOS / "bin" / f"rubberband-{RB_VERSION_FUENTE}"


def _opciones_pylibrb():
    import pylibrb
    O = pylibrb.Option
    alta_calidad = getattr(O, "PITCH_HIGH_QUALITY", None) or getattr(O, "PitchHighQuality")
    return O.ENGINE_FINER | O.PROCESS_OFFLINE | alta_calidad


def _afinar_pylibrb(x, escala):
    import pylibrb
    canales, n = x.shape
    rb = pylibrb.RubberBandStretcher(sample_rate=SR, channels=canales, options=_opciones_pylibrb(),
                                     initial_time_ratio=1.0, initial_pitch_scale=escala)
    if rb.engine_version != 3:
        raise RuntimeError(f"pylibrb usa el motor {rb.engine_version}, se necesita R3")
    rb.set_expected_input_duration(n)
    rb.set_max_process_size(BLOQUE_RB)
    x = np.ascontiguousarray(x, dtype=np.float32)
    bloques = 2 * math.ceil(n / BLOQUE_RB)  # estudio + proceso, para el avance
    for k, i in enumerate(range(0, n, BLOQUE_RB)):  # 1) estudio (pasada de análisis offline)
        avisar_segmento(k, bloques)
        rb.study(np.ascontiguousarray(x[:, i:i + BLOQUE_RB]), final=i + BLOQUE_RB >= n)
    partes = []
    for k, i in enumerate(range(0, n, BLOQUE_RB)):  # 2) proceso
        avisar_segmento(bloques // 2 + k, bloques)
        rb.process(np.ascontiguousarray(x[:, i:i + BLOQUE_RB]), final=i + BLOQUE_RB >= n)
        partes.append(rb.retrieve_available())
    while not rb.is_done():
        partes.append(rb.retrieve_available())
    y = np.concatenate([p for p in partes if p.size], axis=1)
    return _ajustar_largo(y, n)


def _ajustar_largo(y, n):
    if y.shape[1] >= n:
        return np.ascontiguousarray(y[:, :n], dtype=np.float32)
    return np.pad(y, ((0, 0), (0, n - y.shape[1]))).astype(np.float32)


def compilar_rubberband():
    """Plan B: compila rubberband 3 (programa de línea de comandos) y lo guarda en Drive."""
    if RB_BINARIO.exists():
        return RB_BINARIO
    print("  Compilando Rubber Band", RB_VERSION_FUENTE, "desde el código fuente (~2 min)…")
    subprocess.run(["apt-get", "-qq", "install", "-y", "meson", "ninja-build", "libsndfile1-dev"], check=True)
    carpeta = TRABAJO / "_rubberband_fuente"
    carpeta.mkdir(parents=True, exist_ok=True)
    tar = carpeta / f"rubberband-{RB_VERSION_FUENTE}.tar.bz2"
    subprocess.run(["wget", "-q", "-O", str(tar),
                    f"https://breakfastquay.com/files/releases/rubberband-{RB_VERSION_FUENTE}.tar.bz2"], check=True)
    subprocess.run(["tar", "xjf", str(tar), "-C", str(carpeta)], check=True)
    fuente = carpeta / f"rubberband-{RB_VERSION_FUENTE}"
    subprocess.run(["meson", "setup", "build", "-Dfft=builtin", "-Dresampler=builtin", "-Dcmdline=enabled"],
                   cwd=fuente, check=True, capture_output=True)
    subprocess.run(["ninja", "-C", "build"], cwd=fuente, check=True, capture_output=True)
    RB_BINARIO.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(fuente / "build" / "rubberband", RB_BINARIO)
    borrar_seguro(carpeta)
    return RB_BINARIO


def _afinar_cli(x, escala):
    binario = compilar_rubberband()
    entrada, salida = TRABAJO / "_rb_entrada.wav", TRABAJO / "_rb_salida.wav"
    sf.write(str(entrada), x.T, SR, subtype="FLOAT")
    subprocess.run([str(binario), "--fine", "--pitch-hq", "-q", "-f", repr(escala), str(entrada), str(salida)],
                   check=True)
    y, _ = sf.read(str(salida), dtype="float32", always_2d=True)
    borrar_seguro(entrada)
    borrar_seguro(salida)
    return _ajustar_largo(y.T, x.shape[1])


MOTOR_RB = None  # lo fija la autoprueba: "pylibrb" o "cli"


def afinar(x, cents):
    """Pitch-shift de `cents` sin cambiar el tempo, con el motor que pasó la autoprueba."""
    escala = 2 ** (cents / 1200)
    return _afinar_pylibrb(x, escala) if MOTOR_RB == "pylibrb" else _afinar_cli(x, escala)


def _tono_prueba(f0, segundos=8.0):
    t = np.arange(int(segundos * SR)) / SR
    y = 0.3 * np.sin(2 * np.pi * f0 * t) + 0.1 * np.sin(2 * np.pi * 2 * f0 * t) + 0.05 * np.sin(2 * np.pi * 3 * f0 * t)
    return np.stack([y, y]).astype(np.float32)


def autoprueba_rubberband(motor):
    """Tono de 440 Hz desplazado +100 ¢: la frecuencia medida debe quedar a ≤ 0,5 ¢ de 466,16 Hz."""
    global MOTOR_RB
    MOTOR_RB = motor
    x = _tono_prueba(440.0)
    y = afinar(x, 100.0)
    esperado = 440.0 * 2 ** (100 / 1200)
    error = 1200 * math.log2(frecuencia_dominante(y) / esperado)
    return error, y.shape == x.shape


for _motor in ("pylibrb", "cli"):
    try:
        _err, _largo_ok = autoprueba_rubberband(_motor)
        if abs(_err) <= 0.5 and _largo_ok:
            print(f"✓ Rubber Band R3 vía {_motor}: autoprueba +100 ¢ con error {_err:+.3f} ¢")
            break
        print(f"⚠ {_motor}: autoprueba fallida (error {_err:+.3f} ¢, largo correcto: {_largo_ok})")
    except Exception as e:
        print(f"⚠ {_motor} no disponible: {e}")
    MOTOR_RB = None
if MOTOR_RB is None and AFINAR:
    raise Detenido("Ningún motor de Rubber Band pasó la autoprueba: no se puede afinar. "
                   "Desmarca AFINAR en la configuración para separar sin afinar, o vuelve a intentar más tarde.")


## 6 · Modelos
Catálogo de audio-separator (resuelto con su lista oficial), modelos de guitarra desde HuggingFace (`noblebarkrr/mvsepless_resources`, rutas explícitas) y cuatro arreglos en memoria: `mlp_expansion_factor`, carga fuera del catálogo, modelos de más de 16 stems y avance de Demucs por segmento.

In [ ]:
#@title 6 · Modelos: catálogo, guitarras desde HuggingFace y arreglos de audio-separator
import difflib, gc, logging
from importlib.metadata import version as _version

if _version("audio-separator") != "0.44.5":
    raise Detenido(f"Se esperaba audio-separator 0.44.5 y hay {_version('audio-separator')}. "
                   "El arreglo de MelBandRoformer depende de esa versión: reinstala 0.44.5.")

from audio_separator.separator import Separator
import audio_separator.separator.uvr_lib_v5.roformer.mel_band_roformer as _mbr

# ---------- Arreglo 1 (monkeypatch, solo en memoria): mlp_expansion_factor ----------
# En 0.44.5, MelBandRoformer.__init__ recibe mlp_expansion_factor pero NO se lo pasa a su
# MaskEstimator, que usa siempre 4. Eso rompe becruily (factor 1) y listra92 (factor 2).
# BS-Roformer no tiene el bug. No se toca site-packages.
if not getattr(_mbr.MelBandRoformer, "_arreglo_mlp", False):
    _MaskEstimatorOriginal = _mbr.MaskEstimator
    _factor_actual = [4]

    class _MaskEstimatorArreglado(_MaskEstimatorOriginal):
        def __init__(self, dim, dim_inputs, depth, mlp_expansion_factor=None):
            if mlp_expansion_factor is None:
                mlp_expansion_factor = _factor_actual[0]
            super().__init__(dim=dim, dim_inputs=dim_inputs, depth=depth, mlp_expansion_factor=mlp_expansion_factor)

    _init_original = _mbr.MelBandRoformer.__init__

    def _init_arreglado(self, *args, mlp_expansion_factor=4, **kwargs):
        _factor_actual[0] = mlp_expansion_factor
        try:
            _init_original(self, *args, mlp_expansion_factor=mlp_expansion_factor, **kwargs)
        finally:
            _factor_actual[0] = 4

    _mbr.MaskEstimator = _MaskEstimatorArreglado
    _mbr.MelBandRoformer.__init__ = _init_arreglado
    _mbr.MelBandRoformer._arreglo_mlp = True

# ---------- Arreglo 2: cargar checkpoints que no están en el catálogo ----------
# El CLI (-m) los rechaza aunque el archivo exista. Se usa la API interna y, para los
# modelos propios, se salta la búsqueda en el catálogo devolviendo la ruta local.
MODELOS_PROPIOS = {}  # nombre_archivo.ckpt -> nombre_archivo.yaml
if not getattr(Separator, "_arreglo_catalogo", False):
    _descarga_original = Separator.download_model_files

    def _descarga_arreglada(self, model_filename):
        if model_filename in MODELOS_PROPIOS:
            ruta = os.path.join(self.model_file_dir, model_filename)
            return model_filename, "MDXC", Path(model_filename).stem, ruta, MODELOS_PROPIOS[model_filename]
        return _descarga_original(self, model_filename)

    Separator.download_model_files = _descarga_arreglada
    Separator._arreglo_catalogo = True

# ---------- Arreglo 3: modelos con más de 16 stems ----------
# El validador de 0.44.5 limita num_stems a 1–16 y rechaza el BS-Roformer "mega" de
# 53 stems (usado para la guitarra acústica). Se amplía el rango solo en memoria.
from audio_separator.separator.roformer.parameter_validator import ParameterValidator as _Validador
if _Validador.PARAMETER_RANGES.get("num_stems", (1, 64))[1] < 64:
    _Validador.PARAMETER_RANGES = {**_Validador.PARAMETER_RANGES, "num_stems": (1, 64)}

# ---------- Arreglo 4 (solo visual): avance de Demucs por segmento ----------
# Demucs dibuja su propia barra tqdm por cada pasada; se reemplaza en memoria por un
# envoltorio que informa "segmento k de M" al avance del notebook, sin imprimir líneas.
import types as _types
import audio_separator.separator.uvr_lib_v5.demucs.apply as _demucs_apply


def _iterar_con_avance(iterable, *args, **kwargs):
    elementos = list(iterable)
    for k, e in enumerate(elementos):
        avisar_segmento(k, len(elementos))
        yield e
    avisar_segmento(len(elementos), len(elementos))


_demucs_apply.tqdm = _types.SimpleNamespace(tqdm=_iterar_con_avance)

# ---------- Modelos del catálogo (resueltos con la lista oficial, nunca de memoria) ----------
_sep_info = Separator(info_only=True, log_level=logging.WARNING, model_file_dir=str(MODELOS))
_catalogo = set()
for _tipo, _modelos in _sep_info.list_supported_model_files().items():
    for _nombre, _info in _modelos.items():
        _catalogo.add(_info["filename"])
        _catalogo.update(f.split("/")[-1] for f in _info.get("download_files", []))


def resolver_catalogo(que, candidatos):
    for c in candidatos:
        if c in _catalogo:
            return c
    parecidos = difflib.get_close_matches(candidatos[0], sorted(_catalogo), n=8, cutoff=0.3)
    raise Detenido(f"No encontré el modelo de {que} en el catálogo de audio-separator 0.44.5.\n"
                   f"Busqué: {candidatos}\nParecidos: {parecidos}")


MODELO_VOZ = resolver_catalogo("voz (BS-RoFormer)", [
    "model_bs_roformer_ep_317_sdr_12.9755.ckpt", "model_bs_roformer_ep_368_sdr_12.9628.ckpt"])
MODELO_KARAOKE = resolver_catalogo("karaoke (Mel-Band RoFormer)", [
    "mel_band_roformer_karaoke_becruily.ckpt", "mel_band_roformer_karaoke_aufr33_viperx_sdr_10.1956.ckpt",
    "mel_band_roformer_karaoke_gabox.ckpt"])
MODELO_DEMUCS = resolver_catalogo("Demucs 6 stems", ["htdemucs_6s.yaml"])

# ---------- Modelos de guitarra (comunidad MSST, HuggingFace) ----------
# Rutas EXPLÍCITAS verificadas en el repo (no hay models.json). El nombre local en
# Drive/stems/modelos contiene "roformer" y el tipo (mel_band_ / bs_): audio-separator
# detecta la arquitectura por el nombre del archivo.
REPO_HF = "noblebarkrr/mvsepless_resources"
MODELOS_HF = {  # clave: (ckpt en el repo, yaml en el repo, nombre local sin extensión)
    "becruily": ("mel_band_roformer/mbr_guitar_becruily.ckpt",
                 "mel_band_roformer/mbr_guitar_becruily_config.yaml",
                 "mel_band_roformer_guitar_becruily"),
    "chencfd": ("mel_band_roformer/mbr_guitar_chencfd.ckpt",
                "mel_band_roformer/mbr_guitar_chencfd_config.yaml",
                "mel_band_roformer_guitar_chencfd"),
    "punteo": ("mel_band_roformer/mbr_lead_rhythm_guitar_listra92.ckpt",
               "mel_band_roformer/mbr_lead_rhythm_guitar_listra92_config.yaml",
               "mel_band_roformer_lead_rhythm_guitar_listra92"),
    # acústica: BS-Roformer "mega" de MVSep, 53 stems (1,37 GB). Solo se usa su stem
    # 'acoustic-guitar'; las otras 52 salidas se descartan sin guardarlas.
    "acustica": ("bs_roformer/bs_mega_53stem_full_mvsep.ckpt",
                 "bs_roformer/bs_mega_53stem_full_mvsep_config.yaml",
                 "bs_roformer_mega_53stem_mvsep"),
}
_archivos_repo = []


def preparar_modelo_hf(que):
    """Descarga (una sola vez, a Drive/stems/modelos) y registra un modelo de HuggingFace."""
    ckpt_repo, yaml_repo, local = MODELOS_HF[que]
    ckpt_local, yaml_local = MODELOS / f"{local}.ckpt", MODELOS / f"{local}.yaml"
    if not (ckpt_local.exists() and yaml_local.exists()):
        from huggingface_hub import hf_hub_download, list_repo_files
        if not _archivos_repo:
            _archivos_repo.extend(list_repo_files(REPO_HF))
        faltan = [r for r in (ckpt_repo, yaml_repo) if r not in _archivos_repo]
        if faltan:
            con_guitar = [f for f in _archivos_repo if "guitar" in f.lower()]
            raise Detenido(f"No están en {REPO_HF}: {faltan}\nArchivos del repo con 'guitar':\n  "
                           + "\n  ".join(con_guitar))
        descargas = TRABAJO / "_descargas_hf"
        for repo, destino in ((yaml_repo, yaml_local), (ckpt_repo, ckpt_local)):
            print(f"    descargando {repo} → {destino.name}")
            temporal = hf_hub_download(REPO_HF, repo, local_dir=str(descargas))
            parcial = destino.with_name(destino.name + ".parcial")
            shutil.copyfile(temporal, parcial)
            os.replace(parcial, destino)
        borrar_seguro(descargas)  # copia temporal local (dentro de /content/trabajo)
    MODELOS_PROPIOS[ckpt_local.name] = yaml_local.name
    return ckpt_local.name


MODELO_GUITARRAS = MODELO_PUNTEO = MODELO_ACUSTICA = None
if MODO == "completo":
    MODELO_GUITARRAS = preparar_modelo_hf(MODELO_GUITARRA)
    MODELO_PUNTEO = preparar_modelo_hf("punteo")
    if SEPARAR_ACUSTICA_ELECTRICA:
        MODELO_ACUSTICA = preparar_modelo_hf("acustica")

print("✓ Modelos:")
for _k, _v in [("voz", MODELO_VOZ), ("karaoke", MODELO_KARAOKE), ("demucs", MODELO_DEMUCS),
               ("guitarras", MODELO_GUITARRAS), ("punteo", MODELO_PUNTEO), ("acústica", MODELO_ACUSTICA)]:
    if _v:
        print(f"   {_k:10s} {_v}")


## 7 · Separación en cascada
Cada etapa: extraído + resto (= entrada − extraído). Los RoFormer corren con un bucle propio que guarda solo los stems pedidos (el modelo de 53 stems no cabe completo en la RAM).

In [ ]:
#@title 7 · Separación en cascada con restas exactas
# PRINCIPIO: cada etapa entrega "lo extraído" (salida del modelo) y "el resto" =
# entrada − extraído, por RESTA EXACTA en float32 sobre la misma señal. Nunca se usa una
# segunda salida independiente del modelo. Así la suma de todos los stems = la mezcla.
#
# Para no romper la resta, los modelos se llaman directo, sin el camino normal de
# audio-separator (separate), que normaliza entrada y salidas y las cuantiza al escribir.
# RoFormer: bucle de inferencia propio por segmentos (ventana de Hamming + solape), que
# guarda SOLO los stems pedidos: el modelo "mega" de 53 stems no cabe en la RAM de Colab
# si se acumulan todas sus salidas. Si la GPU se queda sin memoria, se reintenta con
# segmentos de la mitad (mínimo SEGMENTO_MIN_S) y el valor usado queda en el manifiesto.
SOLAPAMIENTO_MAX = 4      # tope de pasos por segmento (num_overlap del yaml, si es menor)
SEGMENTO_MIN_S = 2.0

# Qué salida del modelo es "lo extraído" en cada etapa (palabras clave del nombre del stem).
# Si el modelo solo entrega el complemento, extraído = entrada − complemento (sigue exacto).
# "exacto": nombre exacto del stem (para modelos con muchos stems parecidos).
ETAPAS = {
    "voz":           {"objetivo": ["vocal"], "complemento": ["instrument", "other"]},
    "voz_principal": {"objetivo": ["lead", "vocal"], "complemento": ["karaoke", "back", "instrument", "other"]},
    "guitarras":     {"objetivo": ["guitar"], "complemento": ["other", "rest", "instrument"]},
    "punteo":        {"objetivo": ["lead"], "complemento": ["rhythm", "other"]},
    # modelo mega de 53 stems: solo 'acoustic-guitar' (no 'guitar' ni 'electric-guitar')
    "acustica":      {"exacto": "acoustic-guitar"},
}
NOMBRES_ETAPA = {"voz": "voz (BS-RoFormer)", "voz_principal": "voz principal / coros (karaoke)",
                 "guitarras": "guitarras", "punteo": "punteo / rítmica (listra92)",
                 "acustica": "acústica / eléctrica (BS-Roformer mega)",
                 "demucs": "batería, bajo, piano (Demucs 6 stems)"}
_EXCLUYE = ["no_", "no-", "without", "other", "rest", "instrument", "karaoke", "back", "rhythm", "electric"]


def elegir_salida(salidas, entrada, etapa):
    """Devuelve (extraído, descripción) a partir del dict de salidas del modelo."""
    reglas = ETAPAS[etapa]
    if "exacto" in reglas:
        if reglas["exacto"] not in salidas:
            raise RuntimeError(f"Etapa {etapa}: el modelo no entregó '{reglas['exacto']}' (entregó {sorted(salidas)})")
        return salidas[reglas["exacto"]], f"salida '{reglas['exacto']}'"
    for palabra in reglas["objetivo"]:
        for k, v in salidas.items():
            if palabra in k and not any(e in k for e in _EXCLUYE if e != palabra):
                return v, f"salida '{k}'"
    for palabra in reglas["complemento"]:
        for k, v in salidas.items():
            if palabra in k:
                return entrada - v, f"entrada − salida '{k}'"
    raise RuntimeError(f"Etapa {etapa}: no reconozco las salidas del modelo {list(salidas)}")


def _igualar_largo(y, n):
    y = np.asarray(y, dtype=np.float32)
    if y.shape[-1] >= n:
        return np.ascontiguousarray(y[..., :n])
    return np.pad(y, ((0, 0), (0, n - y.shape[-1])))


def cargar_modelo(archivo):
    sep = Separator(log_level=logging.WARNING, model_file_dir=str(MODELOS),
                    output_dir=str(TRABAJO / "_separador"), sample_rate=SR)
    try:
        sep.load_model(archivo)
    except SystemExit:  # audio-separator llama sys.exit(1) si falla la carga: no detener todo
        raise RuntimeError(f"audio-separator no pudo cargar {archivo} (ver mensajes arriba)")
    return sep


def liberar_modelo(sep):
    sep.model_instance = None
    del sep
    gc.collect()
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def nombres_stems(modelo, datos):
    """Nombres de las salidas del RoFormer, en el orden en que las entrega."""
    entrenamiento = datos.get("training", {})
    instrumentos = list(entrenamiento.get("instruments") or [])
    objetivo = entrenamiento.get("target_instrument")
    n = int(getattr(modelo, "num_stems", len(instrumentos) or 1))
    if n > 1:
        if len(instrumentos) != n:
            raise RuntimeError(f"El yaml declara {len(instrumentos)} instrumentos y el modelo entrega {n}")
        return instrumentos
    return [objetivo or (instrumentos[0] if instrumentos else "salida")]


def parametros_segmento(datos):
    """(muestras por segmento, pasos de solape) según el yaml del modelo.
    Prioridad: inference.chunk_size (lo que el autor fijó para separar; en el mega 882000 = 20 s)
    > audio.chunk_size (largo de entrenamiento; en el mega 441000) > hop × (dim_t − 1)."""
    audio, inferencia, modelo = datos.get("audio", {}), datos.get("inference", {}), datos.get("model", {})
    seg = inferencia.get("chunk_size") or audio.get("chunk_size")
    if not seg:
        hop = modelo.get("stft_hop_length") or audio.get("hop_length") or 441
        seg = int(hop) * (int(inferencia.get("dim_t") or audio.get("dim_t") or 801) - 1)
    solape = min(SOLAPAMIENTO_MAX, max(1, int(inferencia.get("num_overlap") or SOLAPAMIENTO_MAX)))
    return int(seg), solape


def _inferir_segmentos(modelo, x, indices, seg, solape, dispositivo):
    import torch
    n = x.shape[1]
    X = torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32))
    # Algunos RoFormer devuelven un segmento algo más corto que la entrada (hop del STFT):
    # se suma solo lo devuelto y se rellena el final con ceros para que toda muestra real
    # quede cubierta por al menos un segmento.
    relleno = max(seg - n, 0) + 8192
    X = torch.nn.functional.pad(X, (0, relleno))
    largo = X.shape[1]
    paso = max(1, seg // solape)
    inicios = list(range(0, largo - seg + 1, paso))
    if inicios[-1] != largo - seg:
        inicios.append(largo - seg)
    ventana = torch.hamming_window(seg, periodic=False)
    acum = torch.zeros((len(indices), 2, largo))
    peso = torch.zeros(largo)
    sel = torch.tensor(indices)
    with torch.no_grad():
        for j, i in enumerate(inicios):
            avisar_segmento(j, len(inicios))
            y = modelo(X[:, i:i + seg].unsqueeze(0).to(dispositivo))[0]
            if y.ndim == 2:
                y = y.unsqueeze(0)
            m = min(seg, y.shape[-1])
            acum[..., i:i + m] += y.index_select(0, sel.to(y.device))[..., :m].float().cpu() * ventana[:m]
            peso[i:i + m] += ventana[:m]
            del y
    avisar_segmento(len(inicios), len(inicios))
    if float(peso[:n].min()) <= 0:
        raise RuntimeError("Inferencia por segmentos: quedaron muestras sin cubrir")
    return (acum[..., :n] / peso[:n]).numpy()


def inferir_roformer(inst, x, solo=None, info_segmento=None):
    """Corre un RoFormer cargado por audio-separator sobre x (2, N). Devuelve {nombre: (2, N)}
    solo con los stems de `solo` (o todos). Sin normalizar niveles."""
    import torch
    datos = inst.model_data
    modelo = inst.model_run
    nombres = [str(k) for k in nombres_stems(modelo, datos)]
    pedidos = nombres if solo is None else list(solo)
    faltan = [p for p in pedidos if p not in nombres]
    if faltan:
        raise RuntimeError(f"El modelo no tiene {faltan}; tiene {nombres}")
    indices = [nombres.index(p) for p in pedidos]
    seg, solape = parametros_segmento(datos)
    dispositivo = next(modelo.parameters()).device
    normalizar = bool(datos.get("inference", {}).get("normalize", False))
    media, desv = (float(x.mean()), float(x.std()) or 1.0) if normalizar else (0.0, 1.0)
    entrada = (x - media) / desv if normalizar else x
    while True:
        try:
            salida = _inferir_segmentos(modelo, entrada, indices, seg, solape, dispositivo)
            break
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if seg // 2 < SEGMENTO_MIN_S * SR:
                raise
            seg //= 2
            print(f"    (memoria de GPU insuficiente: reintento con segmentos de {seg / SR:.1f} s)")
    if info_segmento is not None:
        info_segmento.update(muestras=seg, segundos=round(seg / SR, 2), solape=solape)
    if normalizar:
        salida = salida * desv + media
    return {p.lower(): _igualar_largo(salida[k], x.shape[1]) for k, p in enumerate(pedidos)}


def salidas_modelo(sep, x, solo=None, info_segmento=None):
    """Corre el modelo sobre x (2, N) float32 y devuelve {nombre_en_minúsculas: (2, N)} sin tocar niveles."""
    inst = sep.model_instance
    n = x.shape[1]
    if type(inst).__name__ == "DemucsSeparator":
        from audio_separator.separator.architectures import demucs_separator as ds
        ruta = Path(inst.model_path)
        modelo = ds.get_demucs_model(name=ruta.stem, repo=ruta.parent)
        modelo = ds.demucs_segments(inst.segment_size, modelo)
        modelo.to(inst.torch_device).eval()
        inst.demucs_model_instance = modelo
        try:
            fuentes = inst.demix_demucs(x)
        finally:
            inst.demucs_model_instance = None
            del modelo
        mapa = ds.DEMUCS_6_SOURCE_MAPPER if len(fuentes) == 6 else ds.DEMUCS_4_SOURCE_MAPPER
        return {str(nombre).lower(): _igualar_largo(fuentes[i], n) for nombre, i in mapa.items()}
    if getattr(inst, "is_roformer", False):
        return inferir_roformer(inst, x, solo, info_segmento)
    salida = inst.demix(mix=x)
    if not isinstance(salida, dict):
        salida = {str(inst.primary_stem_name): salida}
    return {str(k).lower(): _igualar_largo(v, n) for k, v in salida.items()}


def separador_real(registro, tiempos, segmentos=None):
    """Devuelve separar(etapa, x) usando los modelos reales. Anota en `registro` qué salida usó
    y en `segmentos` el tamaño de segmento usado por cada RoFormer."""
    modelos = {"voz": MODELO_VOZ, "voz_principal": MODELO_KARAOKE, "guitarras": MODELO_GUITARRAS,
               "punteo": MODELO_PUNTEO, "acustica": MODELO_ACUSTICA, "demucs": MODELO_DEMUCS}

    def separar(etapa, x):
        solo = [ETAPAS[etapa]["exacto"]] if "exacto" in ETAPAS.get(etapa, {}) else None
        info_seg = {}
        with Etapa(NOMBRES_ETAPA[etapa], tiempos):
            sep = cargar_modelo(modelos[etapa])
            try:
                salidas = salidas_modelo(sep, x, solo, info_seg)
            finally:
                liberar_modelo(sep)
        if segmentos is not None and info_seg:
            segmentos[etapa] = info_seg
        if etapa == "demucs":
            registro[etapa] = f"{modelos[etapa]}: {sorted(salidas)}"
            return salidas
        extraido, como = elegir_salida(salidas, x, etapa)
        seg_txt = f", segmento {info_seg['segundos']} s" if info_seg else ""
        registro[etapa] = f"{modelos[etapa]}: {como} (salidas usadas: {sorted(salidas)}{seg_txt})"
        return extraido

    return separar


def cascada(mezcla, separar, modo, orden, por_timbre):
    """Separa `mezcla` (2, N) float32. Devuelve (stems, guitarras_partes, guitarras_total).
    stems: dict "NN Nombre" -> array; su suma es exactamente `mezcla` (salvo redondeo float32)."""
    voz = separar("voz", mezcla)
    instrumental = mezcla - voz
    principal = separar("voz_principal", voz)
    coros = voz - principal
    if modo == "karaoke":
        return {"01 Voz principal": principal, "02 Coros": coros, "03 Instrumental": instrumental}, None, None

    guitarras = separar("guitarras", instrumental)
    sin_guitarras = instrumental - guitarras

    if orden == "punteo primero" or not por_timbre:
        punteo = separar("punteo", guitarras)
        ritmicas = guitarras - punteo
        if por_timbre:
            ritm_ac = separar("acustica", ritmicas)
            ritm_el = ritmicas - ritm_ac
            partes = {"05 Guitarra punteo": punteo, "06 Guitarras ritmicas electricas": ritm_el,
                      "07 Guitarras ritmicas acusticas": ritm_ac}
        else:
            partes = {"05 Guitarra punteo": punteo, "06 Guitarras ritmicas": ritmicas}
    else:  # "timbre primero"
        acusticas = separar("acustica", guitarras)
        electricas = guitarras - acusticas
        punteo = separar("punteo", electricas)
        electricas_resto = electricas - punteo
        partes = {"05 Guitarra punteo": punteo, "06 Guitarras electricas": electricas_resto,
                  "07 Guitarras acusticas": acusticas}

    d = separar("demucs", sin_guitarras)
    otros = sin_guitarras - (d["drums"] + d["bass"] + d["piano"])
    stems = {"01 Voz principal": principal, "02 Coros": coros, "03 Bateria": d["drums"], "04 Bajo": d["bass"]}
    stems.update(partes)
    stems["08 Piano"] = d["piano"]
    stems["09 Otros"] = otros
    return stems, partes, guitarras

print("✓ Separación lista")


## 8 · Informes
`afinacion.txt` (con la prueba de cancelación) y `guitarras.txt` (mapa de punteo).

In [ ]:
#@title 8 · Informes: afinacion.txt y guitarras.txt
# --- Mapa de punteo (guitarras.txt): constantes a la vista para ajustarlas ---
VENTANA_RMS_S = 0.5          # largo de cada ventana de energía
HOP_RMS_S = 0.25             # avance entre ventanas
UMBRAL_PUNTEO_REL = 0.25     # punteo activo si su RMS ≥ 0,25 × RMS de guitarras (−12 dB)
PISO_GUITARRAS_DBFS = -50.0  # y las guitarras suenan sobre −50 dBFS
MIN_TRAMO_S = 2.0            # tramo mínimo
UNIR_HUECO_S = 1.0           # se unen tramos separados por menos de esto
RESIDUO_MAX_DBFS = -90.0     # prueba de cancelación (FLAC 24-bit)


def _rms_ventanas(x, sr):
    e = np.mean(np.asarray(x, dtype=np.float64) ** 2, axis=0)
    v, h = int(VENTANA_RMS_S * sr), int(HOP_RMS_S * sr)
    acum = np.concatenate([[0.0], np.cumsum(e)])
    inicios = np.arange(0, max(1, len(e) - v + 1), h)
    return inicios / sr, np.sqrt(np.maximum(acum[inicios + v] - acum[inicios], 0) / v)


def tramos_punteo(punteo, guitarras, sr=SR):
    """Tramos (inicio_s, fin_s) con punteo sostenido."""
    t, rms_p = _rms_ventanas(punteo, sr)
    _, rms_g = _rms_ventanas(guitarras, sr)
    piso = 10 ** (PISO_GUITARRAS_DBFS / 20)
    activo = (rms_p >= UMBRAL_PUNTEO_REL * rms_g) & (rms_g > piso)
    tramos = []
    for i in np.flatnonzero(activo):
        ini, fin = t[i], t[i] + VENTANA_RMS_S
        if tramos and ini - tramos[-1][1] < UNIR_HUECO_S:
            tramos[-1][1] = max(tramos[-1][1], fin)
        else:
            tramos.append([ini, fin])
    dur = punteo.shape[1] / sr
    return [(a, min(b, dur)) for a, b in tramos if min(b, dur) - a >= MIN_TRAMO_S]


def proporciones(partes, desde=0, hasta=None):
    """Proporción de energía de cada parte de guitarra (suma 100 %) en el rango de muestras."""
    e = {k: float(np.sum(np.asarray(v[:, desde:hasta], dtype=np.float64) ** 2)) for k, v in partes.items()}
    total = sum(e.values())
    return {k: (100 * v / total if total > 0 else 0.0) for k, v in e.items()}


def texto_guitarras(tema, partes, guitarras, orden, sr=SR):
    nombres = {k: k[3:] for k in partes}
    tramos = tramos_punteo(partes["05 Guitarra punteo"], guitarras, sr)
    lineas = [f"GUITARRAS — {tema}", "",
              f"Orden usado: {orden}" + ("" if len(partes) > 2 else " (sin corte acústica/eléctrica)"), "",
              "Proporción de energía de guitarras en todo el tema:"]
    for k, v in proporciones(partes).items():
        lineas.append(f"  {nombres[k]:32s} {v:5.1f} %")
    lineas += ["", f"Tramos con punteo sostenido ({len(tramos)}):",
               f"  (punteo ≥ {UMBRAL_PUNTEO_REL:g} × guitarras, guitarras > {PISO_GUITARRAS_DBFS:g} dBFS, "
               f"mínimo {MIN_TRAMO_S:g} s, huecos < {UNIR_HUECO_S:g} s unidos)"]
    if not tramos:
        lineas.append("  ninguno")
    for a, b in tramos:
        p = proporciones(partes, int(a * sr), int(b * sr))
        detalle = " · ".join(f"{nombres[k]} {v:.0f} %" for k, v in p.items())
        lineas.append(f"  {mm_ss(a):>7s} – {mm_ss(b):>7s}  ({b - a:4.1f} s)  {detalle}")
    lineas += ["", "Sirve para ver rápido si detectó el solo real o se llevó un riff.",
               "Nada de esto vale hasta escucharlo contra la canción."]
    return "\n".join(lineas) + "\n"


def _nivel_incertidumbre(unc):
    return "alta repetibilidad" if unc <= 1.5 else ("media" if unc <= 6 else "variación alta del material")


def texto_afinacion(tema, info):
    L = [f"AFINACIÓN — {tema}", ""]
    L.append(f"Entrada: {info['archivo']} (frecuencia nativa {info['sr_nativa']} Hz; se trabaja a 44100 Hz)")
    m = info.get("medicion")
    if info["destino_hz"] is None:
        L += ["Afinación: no se midió ni se corrigió (AFINAR desactivado)."]
    else:
        dev_dest = 1200 * math.log2(m["ref_hz"] / info["destino_hz"])
        L += [f"Destino: {info['destino_hz']:.2f} Hz · umbral ±{UMBRAL_CENTS:g} ¢", "",
              "Medición sobre la mezcla sin batería (mezcla − batería de Demucs), método de centrail:",
              f"  Referencia detectada:        {m['ref_hz']:.2f} Hz",
              f"  Desviación respecto de A440: {m['offset']:+.2f} ¢",
              f"  Desviación respecto destino: {dev_dest:+.2f} ¢",
              f"  Incertidumbre:               ±{m['unc']:.2f} ¢ ({_nivel_incertidumbre(m['unc'])}; ventanas pares vs. impares)",
              f"  Consistencia tonal (R):      {100 * m['R']:.1f} % (informativo)",
              f"  Deriva máxima:               {m['deriva_max']:.2f} ¢ (10 segmentos)",
              "  Segmentos (¢ vs A440):       " + " ".join("—" if s is None else f"{s:+.1f}" for s in m["segmentos"]),
              f"  Ventanas analizadas:         {m['ventanas']} (excluidas sin contenido tonal: {m['excluidas']})"]
        if m["deriva_max"] > 10:
            L.append("  ⚠ ADVERTENCIA: la deriva supera 10 ¢; una corrección global no deja todo el tema afinado.")
        L.append("")
        if info["corregido"]:
            L.append(f"Corrección: {info['shift']:+.2f} ¢ aplicada UNA vez a la mezcla original "
                     f"con Rubber Band R3 ({info['motor']}). Todo lo demás sale de la mezcla afinada.")
        else:
            L.append(f"Corrección: ya estaba afinado (|{-info['shift']:+.2f} ¢| ≤ {UMBRAL_CENTS:g} ¢), no se tocó.")
        if info.get("ganancia_afinar_db", 0.0) < -0.01:
            L.append(f"Nivel: se redujo {-info['ganancia_afinar_db']:.2f} dB tras afinar para no superar "
                     f"{'0 dBFS' if info['pico_original'] >= 1 else 'el nivel del original'}.")
        v = info.get("verificacion")
        if v:
            L.append(f"Verificación tras separar (mezcla afinada − batería): {v['ref_hz']:.2f} Hz, "
                     f"{1200 * math.log2(v['ref_hz'] / info['destino_hz']):+.2f} ¢ respecto del destino "
                     f"(±{v['unc']:.2f} ¢).")
    L += ["", "Prueba de cancelación (mezcla de referencia − suma de stems, releyendo los archivos escritos):",
          f"  Referencia: {'00 Mezcla afinada' if info['corregido'] else 'mezcla original'}"
          f" · {FORMATO} · {SR_SALIDA} Hz"]
    if info.get("ganancia_recorte_db", 0.0) < -0.01:
        L.append(f"  Nota: para evitar recorte se bajaron {-info['ganancia_recorte_db']:.2f} dB por igual a todos los "
                 "stems y a la referencia (la suma sigue igual a la mezcla × esa ganancia).")
    L.append(f"  Residuo pico: {info['residuo_dbfs']:.1f} dBFS → {info['veredicto_cancelacion']}")
    L += ["", "Modelos y salidas usadas:"] + [f"  {k}: {v}" for k, v in info["modelos"].items()]
    L += ["", "Tiempos (s): " + ", ".join(f"{k} {v:g}" for k, v in info["tiempos"].items())]
    return "\n".join(L) + "\n"

print("✓ Informes listos")


## 9 · Procesar
Todos los temas de la carpeta de entrada. El avance se ve en las barras, en una línea fija con la etapa actual y en `salida/progreso.txt` (para mirarlo desde otro dispositivo). Si un tema falla, se registra en `salida/errores.txt` y se sigue con el siguiente.

In [ ]:
#@title 9 · Procesar todos los temas de la carpeta de entrada
# Avance: barras (temas / etapas / segmentos), una línea fija con la etapa actual y
# Drive/stems/salida/progreso.txt, que se reescribe en cada etapa (y queda con el resumen).
MOSTRAR_AVANCE = True


def destino_hz():
    if DESTINO_HZ is None:
        return None
    if not 380.0 <= DESTINO_HZ <= 500.0:
        raise Detenido(f"La referencia {DESTINO_HZ} Hz no parece válida (usa algo entre 380 y 500 Hz).")
    return DESTINO_HZ


def preparar_afinacion(mezcla, info, tiempos):
    """Mide sobre mezcla − batería y, si hace falta, corrige la MEZCLA ORIGINAL una sola vez.
    Devuelve la mezcla con la que trabaja todo lo demás (afinada u original)."""
    info.update(destino_hz=destino_hz(), corregido=False, shift=0.0, motor=MOTOR_RB, pico_original=pico(mezcla))
    if info["destino_hz"] is None:
        return mezcla
    with Etapa("medir afinación (mezcla − batería)", tiempos):
        sep = cargar_modelo(MODELO_DEMUCS)
        try:
            bateria = salidas_modelo(sep, mezcla)["drums"]
        finally:
            liberar_modelo(sep)
        info["medicion"] = medicion = medir_afinacion(mezcla - bateria)
        del bateria  # el audio sin batería solo sirve para medir
    info["shift"] = shift = 1200 * math.log2(info["destino_hz"] / medicion["ref_hz"])
    print(f"    {medicion['ref_hz']:.2f} Hz detectado (±{medicion['unc']:.2f} ¢, deriva {medicion['deriva_max']:.1f} ¢)"
          f" → corrección {shift:+.2f} ¢")
    if abs(shift) <= UMBRAL_CENTS:
        print("    ya estaba afinado: no se toca")
        return mezcla
    with Etapa(f"afinar {shift:+.2f} ¢ (Rubber Band R3)", tiempos):
        afinada = afinar(mezcla, shift)
    techo = min(1.0, info["pico_original"])
    p = pico(afinada)
    ganancia = techo / p if p > techo else 1.0
    if ganancia < 1.0:
        afinada = (afinada * ganancia).astype(np.float32)
    info.update(corregido=True, ganancia_afinar_db=dbfs(ganancia))
    return afinada


def escribir_y_verificar(salidas, referencia, nombres_suma, carpeta, info, tiempos):
    """Remuestrea (si toca), evita recorte con UNA ganancia común, escribe y hace la prueba de cancelación."""
    with Etapa(f"escribir {len(salidas)} archivos ({FORMATO}, {SR_SALIDA} Hz)", tiempos):
        ref = a_frecuencia_salida(referencia)
        listos = {k: a_frecuencia_salida(v) for k, v in salidas.items()}
        p = max(pico(v) for v in listos.values())
        techo = 1.0 - 2.0 ** -23
        g = techo / p if p > techo else 1.0
        info["ganancia_recorte_db"] = dbfs(g)
        rutas = {}
        for j, (k, v) in enumerate(listos.items()):
            avisar_segmento(j, len(listos))
            rutas[k] = escribir_audio(carpeta / k, v * g if g < 1 else v, SR_SALIDA)
        avisar_segmento(len(listos), len(listos))
        ref = ref.astype(np.float64) * g
        del listos
    with Etapa("prueba de cancelación", tiempos):
        suma = None
        for j, k in enumerate(nombres_suma):
            avisar_segmento(j + 1, len(nombres_suma))
            y = leer_escrito(rutas[k])
            if suma is None:
                suma = y
            else:
                n = min(suma.shape[1], y.shape[1])
                suma = suma[:, :n] + y[:, :n]
        n = min(suma.shape[1], ref.shape[1])
        residuo = dbfs(pico(suma[:, :n] - ref[:, :n]))
    info["residuo_dbfs"] = residuo
    if ES_MP3:
        info["veredicto_cancelacion"] = "MP3: la suma no es exacta por diseño (solo para escuchar)"
    elif residuo < RESIDUO_MAX_DBFS:
        info["veredicto_cancelacion"] = f"OK (bajo {RESIDUO_MAX_DBFS:g} dBFS)"
    else:
        info["veredicto_cancelacion"] = f"⚠ ADVERTENCIA: sobre {RESIDUO_MAX_DBFS:g} dBFS, la suma no cuadra"
    print(f"    residuo de cancelación: {residuo:.1f} dBFS · {info['veredicto_cancelacion']}")
    return rutas


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in carpeta.iterdir() if f.is_file()) / 1e6


def procesar_tema(ruta):
    tema = ruta.stem
    destino = SALIDA / tema
    manifiesto = destino / ".proceso.json"
    if manifiesto.exists():
        datos = json.loads(manifiesto.read_text(encoding="utf-8"))
        print("  ya estaba completo: se salta (para rehacerlo, borra a mano su carpeta de salida)")
        return {**datos["resumen"], "estado": "ya estaba"}

    t0 = time.time()
    tiempos, registro, segmentos = {}, {}, {}
    trabajo = TRABAJO / tema
    if trabajo.exists():
        borrar_seguro(trabajo)
    trabajo.mkdir(parents=True)
    try:
        with Etapa("decodificar", tiempos):
            mezcla, sr_nativa = leer_audio(ruta)
        info = {"archivo": ruta.name, "sr_nativa": sr_nativa, "modelos": registro, "tiempos": tiempos}
        mezcla_trabajo = preparar_afinacion(mezcla, info, tiempos)
        del mezcla

        stems, partes, guitarras = cascada(mezcla_trabajo, separador_real(registro, tiempos, segmentos),
                                           MODO, ORDEN_GUITARRAS, SEPARAR_ACUSTICA_ELECTRICA)
        if info["corregido"] and MODO == "completo":
            with Etapa("verificar afinación tras separar", tiempos):
                info["verificacion"] = medir_afinacion(mezcla_trabajo - stems["03 Bateria"])

        salidas = ({"00 Mezcla afinada": mezcla_trabajo} if info["corregido"] else {}) | stems
        escribir_y_verificar(salidas, mezcla_trabajo, list(stems), trabajo, info, tiempos)
        if partes is not None:
            (trabajo / "guitarras.txt").write_text(
                texto_guitarras(tema, partes, guitarras, ORDEN_GUITARRAS), encoding="utf-8")
        del stems, partes, guitarras, salidas
        (trabajo / "afinacion.txt").write_text(texto_afinacion(tema, info), encoding="utf-8")
        if CLIC != "no":
            print("    (los clics llegan en la fase 2)")

        with Etapa("copiar a Drive", tiempos):
            destino.mkdir(parents=True, exist_ok=True)
            for f in sorted(trabajo.iterdir()):
                if f.is_file():
                    shutil.copy2(f, destino / f.name)
        m = info.get("medicion")
        resumen = {"tema": tema, "estado": "listo",
                   "hz": round(m["ref_hz"], 2) if m else None,
                   "correccion_cents": round(info["shift"], 2) if info["corregido"] else 0.0,
                   "residuo_dbfs": round(info["residuo_dbfs"], 1),
                   "mb": round(tamano_mb(destino), 1), "minutos": round((time.time() - t0) / 60, 1)}
        manifiesto.write_text(json.dumps({  # se escribe al ÚLTIMO: marca el tema como completo
            "resumen": resumen, "modo": MODO, "orden_guitarras": ORDEN_GUITARRAS,
            "acustica_electrica": SEPARAR_ACUSTICA_ELECTRICA, "formato": FORMATO, "frecuencia": SR_SALIDA,
            "afinar_hz": DESTINO_HZ, "modelos": registro, "segmentos_roformer": segmentos, "fecha": time.strftime("%Y-%m-%d %H:%M")},
            ensure_ascii=False, indent=2), encoding="utf-8")
        return resumen
    finally:
        borrar_seguro(trabajo)


def etapas_previstas():
    """Cuántas etapas tendrá un tema con esta configuración (para la barra de etapas)."""
    n = 1 + 2 + 3  # decodificar · voz, karaoke · escribir, cancelación, copiar
    if DESTINO_HZ is not None:
        n += 2  # medir y (si hace falta) afinar
    if MODO == "completo":
        n += 3 + (1 if SEPARAR_ACUSTICA_ELECTRICA else 0)  # guitarras, punteo, [acústica], demucs
        n += 1 if DESTINO_HZ is not None else 0  # verificar afinación tras separar
    return n


def texto_resumen(filas):
    L = [f"{'tema':32s} {'estado':10s} {'Hz':>8s} {'corr. ¢':>8s} {'residuo dBFS':>13s} {'MB':>7s} {'min':>5s}"]
    for r in filas:
        if r["estado"].startswith("ERROR"):
            L.append(f"{r['tema'][:32]:32s} {r['estado']}")
            continue
        hz = f"{r['hz']:.2f}" if r.get("hz") else "—"
        aviso = " ⚠" if (r["residuo_dbfs"] >= RESIDUO_MAX_DBFS and not ES_MP3) else ""
        L.append(f"{r['tema'][:32]:32s} {r['estado']:10s} {hz:>8s} {r['correccion_cents']:>+8.2f} "
                 f"{r['residuo_dbfs']:>13.1f}{aviso} {r['mb']:>7.1f} {r.get('minutos', 0):>5.1f}")
    L.append("Tramos de tempo: llegan con los clics (fase 2).")
    return "\n".join(L) + "\n"


RESUMEN = []
PROGRESO = Progreso(len(TEMAS), SALIDA / "progreso.txt", mostrar=MOSTRAR_AVANCE)
for _i, _ruta in enumerate(TEMAS, start=1):
    print(f"\n━━━ {_i}/{len(TEMAS)} · {_ruta.name} ━━━")
    PROGRESO.nuevo_tema(_i, _ruta.stem, etapas_previstas())
    try:
        RESUMEN.append(procesar_tema(_ruta))
    except Detenido:
        raise
    except Exception as e:
        traceback.print_exc()
        RESUMEN.append({"tema": _ruta.stem, "estado": f"ERROR: {type(e).__name__}: {e}"})
        with open(SALIDA / "errores.txt", "a", encoding="utf-8") as _f:
            _f.write(f"{time.strftime('%Y-%m-%d %H:%M')} · {_ruta.name}\n{traceback.format_exc()}\n")
        print("  ✗ falló este tema (detalle en salida/errores.txt); sigo con el siguiente")
    PROGRESO.fin_tema()
    gc.collect()

_texto = texto_resumen(RESUMEN)
PROGRESO.cerrar(_texto)
PROGRESO = None
print("\n" + "═" * 96)
print(_texto)


## 10 · (Opcional) Descargar un tema

In [ ]:
#@title 10 · (Opcional) Descargar un tema como .zip
TEMA_A_DESCARGAR = ""  #@param {type:"string"}
#@markdown Escribe el nombre de la carpeta del tema (tal como aparece en salida/) y ejecuta solo esta celda.
if TEMA_A_DESCARGAR.strip():
    _carpeta = SALIDA / TEMA_A_DESCARGAR.strip()
    if not _carpeta.is_dir():
        print(f"No existe {_carpeta}. Temas disponibles:")
        for _c in sorted(p.name for p in SALIDA.iterdir() if p.is_dir()):
            print("   ·", _c)
    else:
        _zip = shutil.make_archive(str(TRABAJO / _carpeta.name), "zip", root_dir=_carpeta)
        from google.colab import files
        files.download(_zip)
        print("✓ Descarga iniciada:", Path(_zip).name)
